In [1]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import math
from sklearn.model_selection import train_test_split

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Example followed at Keras.io

This code is based off the example at [keras.io](https://keras.io/examples/vision/mnist_convnet/), with a bit of an adjustment for the change in data. The example on the keras website uses a built-in dataset, whereas this is using the dataset from kaggle. 

## Read in train and test data

The data is in csv format, so it will need to be transformed a bit.

In [2]:
train_df = pd.read_csv("../input/digit-recognizer/train.csv")
test_df = pd.read_csv("../input/digit-recognizer/test.csv")

## Train / Validation split

The test dataset doesn't have labels, so splitting the data for out of sample score calculation.

In [3]:
# train, validation split
X_train, X_validation, y_train, y_validation = train_test_split(train_df.iloc[:,1:], train_df['label'], test_size=0.2, random_state=19)

# Data transformation

* Put the labels from the train and validation dataframes into numpy arrays
* Reshape the train, validation and test dataframes after transforming to numpy arrays.

Note: X_test does not have a label.

In [4]:
# Create the label numpy arrays
y_train = y_train.to_numpy()
y_validation = y_validation.to_numpy()
# Reshape X data
X_train = X_train.to_numpy()
X_train = X_train.reshape(X_train.shape[0], int(math.sqrt(X_train.shape[1])), -1)

X_validation = X_validation.to_numpy()
X_validation = X_validation.reshape(X_validation.shape[0], int(math.sqrt(X_validation.shape[1])), -1)

# transform X_test to the right shape
X_test = test_df.loc[:, "pixel0":].to_numpy()
len_X_test = len(X_test)
X_test = X_test.reshape(len_X_test, 28, 28)

## Take a look at the shapes

In [5]:
print(f"X_train shape: {X_train.shape}")
print(f"X_validation shape: {X_validation.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (33600, 28, 28)
X_validation shape: (8400, 28, 28)
X_test shape: (28000, 28, 28)


In [6]:
# Number of possible classes for output
num_classes = 10
input_shape = (28, 28, 1)

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_validation = X_validation.astype("float32") / 255
X_test = X_test.astype("float32") / 255

print(f"X_train shape before np.expand_dims: {X_train.shape}")
# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_validation = np.expand_dims(X_validation, -1)
X_test = np.expand_dims(X_test, -1)

print(f"X_train shape after np.expand_dims: {X_train.shape}")
print(f"{X_train.shape[0]} train samples")
print(f"{X_validation.shape[0]} validation samples")
print(f"{X_test.shape[0]} test samples")


X_train shape before np.expand_dims: (33600, 28, 28)
X_train shape after np.expand_dims: (33600, 28, 28, 1)
33600 train samples
8400 validation samples
28000 test samples


## Change the labels to categoricals

From an array of class values to a binary matrix with an entry for each category

In [7]:
y_train = keras.utils.to_categorical(y_train, num_classes)
y_validation = keras.utils.to_categorical(y_validation, num_classes)

## Create the model

In [8]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 26, 26, 32)        320       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 13, 13, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 11, 11, 64)        18496     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 5, 5, 64)          0         
_________________________________________________________________
flatten (Flatten)            (None, 1600)              0         
_________________________________________________________________
dropout (Dropout)            (None, 1600)              0         
_________________________________________________________________
dense (Dense)                (None, 10)                1

2022-09-05 17:52:21.982782: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


## Train the model

In [9]:
batch_size = 128
epochs = 15

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

2022-09-05 17:52:22.345222: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/15
237/237 [==============================] - 9s 35ms/step - loss: 0.5120 - accuracy: 0.8442 - val_loss: 0.1406 - val_accuracy: 0.9580
Epoch 2/15
237/237 [==============================] - 8s 33ms/step - loss: 0.1461 - accuracy: 0.9558 - val_loss: 0.0878 - val_accuracy: 0.9714
Epoch 3/15
237/237 [==============================] - 8s 34ms/step - loss: 0.1066 - accuracy: 0.9681 - val_loss: 0.0700 - val_accuracy: 0.9795
Epoch 4/15
237/237 [==============================] - 9s 38ms/step - loss: 0.0849 - accuracy: 0.9746 - val_loss: 0.0589 - val_accuracy: 0.9833
Epoch 5/15
237/237 [==============================] - 8s 34ms/step - loss: 0.0751 - accuracy: 0.9757 - val_loss: 0.0552 - val_accuracy: 0.9827
Epoch 6/15
237/237 [==============================] - 8s 34ms/step - loss: 0.0673 - accuracy: 0.9786 - val_loss: 0.0563 - val_accuracy: 0.9824
Epoch 7/15
237/237 [==============================] - 8s 34ms/step - loss: 0.0610 - accuracy: 0.9803 - val_loss: 0.0508 - val_accuracy: 0.9863

In [10]:
score = model.evaluate(X_validation, y_validation, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.043962422758340836
Test accuracy: 0.986547589302063


## Generate predictions

In [11]:
y_pred = model.predict(X_test)
classes_x=np.argmax(y_pred, axis=1)

In [12]:
submit_this = pd.DataFrame({'Label': classes_x, 'ImageId': range(1, len(classes_x)+1)})
submit_this.to_csv("submission.csv", index=False)